# 21. ODIN and Trust Score: A Real, Same-Protocol Comparison

`STUDY_PLAN.md` 3.6's OOD-detection comparison table previously carried ODIN (Liang et al., 2018) and
Trust Score (Jiang et al., 2018) as **qualitative** claims sourced from their own papers' reported numbers
on different datasets/backbones/splits ("ODIN... comparable-or-worse separation... kept only as an
evaluation baseline"; "Trust Score... likely stronger raw OOD signal... traded away for
architecture-agnosticism") - real methods, but never actually run on this project's own data, so the
comparison was closer to citation than measurement.

This notebook reimplements both from their own papers and runs them on the *exact same* real ResNet-50
images/splits/labels this project's own combiner is evaluated on (`data/logit_cache_resnet50.pt`), producing
a genuine same-protocol head-to-head with bootstrap CIs and DeLong tests (`significance.py`), not a
cited comparison.

- **Trust Score** (`src/deployment_reliability/trust_score.py`) reuses the existing cached penultimate-layer
  ResNet-50 features (`data/mahalanobis_feature_cache_resnet50.pt`) already collected for the Mahalanobis-lite
  check (`DESIGN.md` 25) - no new data collection needed, since Trust Score is a feature-space method like
  Mahalanobis, just with a different (k-NN ratio, not Gaussian) distance model.
- **ODIN** (`scripts/collect_odin_scores.py`) needed a genuinely new data collection run: a real
  forward+backward+forward pass (temperature-scaled loss, FGSM-style input perturbation, second forward pass)
  over the same 7,235 cached images, at the paper's own stated defaults (T=1000, epsilon=0.0014) - deliberately
  not tuned on this project's own data, so this stays a fair "as-published" comparison in both directions.

**A real finding, investigated and reported exactly as found before any comparison numbers were trusted:**
ODIN's paper-specified T=1000, tuned by the original authors for CIFAR-scale networks, nearly flattens
ResNet-50's tempered softmax to uniform (1/C=0.001) for almost every image - this backbone's per-image logit
range (~7.7) is far too small relative to T=1000 to survive the division before exponentiating. This is
disclosed below, not silently worked around.

In [1]:
import os
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
torch.manual_seed(0)

from deployment_reliability.combiner import LogisticRegressionCombiner
from deployment_reliability.features import featurize
from deployment_reliability.router import auroc
from deployment_reliability.significance import bootstrap_auroc_ci, delong_test
from deployment_reliability.trust_score import TrustScorer

DATA_DIR = os.path.join("..", "data")


def mask(splits_arr, name):
    return torch.from_numpy(np.asarray(splits_arr) == name)


# --- Combiner (this project's own score, the thing being compared against) ---
logit_cache = torch.load(os.path.join(DATA_DIR, "logit_cache_resnet50.pt"))
logits, labels, splits_arr = logit_cache["logits"], logit_cache["labels"], logit_cache["splits"]
m_fit = mask(splits_arr, "combiner_fit")
m_test = mask(splits_arr, "id_test")
m_a = mask(splits_arr, "imagenet_a")
m_o = mask(splits_arr, "imagenet_o")
correct = logits.argmax(dim=-1) == labels
phi = featurize(logits)
combiner = LogisticRegressionCombiner().fit(phi[m_fit], correct[m_fit].float())
s_combiner = combiner.score(phi)

# --- Trust Score (feature-space, reuses the Mahalanobis feature cache) ---
feat_cache = torch.load(os.path.join(DATA_DIR, "mahalanobis_feature_cache_resnet50.pt"))
features = feat_cache["features"]
feat_splits = np.asarray(feat_cache["splits"])
fm_fit = torch.from_numpy(feat_splits == "combiner_fit")
trust_scorer = TrustScorer().fit(features[fm_fit], feat_cache["labels"][fm_fit])
predicted_all = feat_cache["logits"].argmax(dim=-1)
s_trust = trust_scorer.score(features, predicted_all)

# --- ODIN (its own cache, own logit ordering - same files/splits, verified aligned below) ---
odin_cache = torch.load(os.path.join(DATA_DIR, "odin_cache_resnet50.pt"))
s_odin = odin_cache["scores"]
odin_splits = np.asarray(odin_cache["splits"])

# Alignment check: all three caches were built from data/logit_cache_resnet50.pt's own file
# list (collect_features_mahalanobis.py and collect_odin_scores.py both load it directly and
# never re-sample) - confirm the split arrays actually match position-for-position before
# trusting any cross-cache comparison below.
assert list(feat_splits) == list(splits_arr), "mahalanobis feature cache split order diverges from logit cache"
assert list(odin_splits) == list(splits_arr), "ODIN cache split order diverges from logit cache"
assert torch.equal(feat_cache["labels"], labels) and torch.equal(odin_cache["labels"], labels)

print(f"combiner_fit n={int(m_fit.sum())}  id_test n={int(m_test.sum())}  imagenet_a n={int(m_a.sum())}  imagenet_o n={int(m_o.sum())}")
print(f"ODIN temperature={odin_cache['temperature']}  epsilon={odin_cache['epsilon']}")

combiner_fit n=1500  id_test n=1500  imagenet_a n=1935  imagenet_o n=2000
ODIN temperature=1000.0  epsilon=0.0014


## Comparison 1 — `corr_id`: does each score separate correct from incorrect `id_test` predictions?

All three scores are evaluated on the exact same `id_test` images against the exact same correctness
labels — the textbook DeLong scenario (paired, correlated scores over identical samples), not two
independent samples.

In [2]:
correct_test = correct[m_test]
s_combiner_test = s_combiner[m_test]
s_trust_test = s_trust[m_test]
s_odin_test = s_odin[m_test]

delong_trust = delong_test(correct_test, s_combiner_test, s_trust_test)
delong_odin = delong_test(correct_test, s_combiner_test, s_odin_test)

ci_combiner = bootstrap_auroc_ci(s_combiner_test[correct_test], s_combiner_test[~correct_test], n_bootstrap=3000, seed=0)
ci_trust = bootstrap_auroc_ci(s_trust_test[correct_test], s_trust_test[~correct_test], n_bootstrap=3000, seed=0)
ci_odin = bootstrap_auroc_ci(s_odin_test[correct_test], s_odin_test[~correct_test], n_bootstrap=3000, seed=0)

print("=== corr_id AUROC (95% bootstrap CI) ===")
print(f"Combiner:    {ci_combiner.auroc:.4f}  [{ci_combiner.ci_lo:.4f}, {ci_combiner.ci_hi:.4f}]")
print(f"Trust Score: {ci_trust.auroc:.4f}  [{ci_trust.ci_lo:.4f}, {ci_trust.ci_hi:.4f}]")
print(f"ODIN:        {ci_odin.auroc:.4f}  [{ci_odin.ci_lo:.4f}, {ci_odin.ci_hi:.4f}]")
print()
print("=== DeLong paired tests vs. combiner ===")
print(f"Combiner vs. Trust Score: diff={delong_trust.auc_diff:+.4f}  z={delong_trust.z:.3f}  p={delong_trust.p_value:.2e}")
print(f"Combiner vs. ODIN:        diff={delong_odin.auc_diff:+.4f}  z={delong_odin.z:.3f}  p={delong_odin.p_value:.2e}")

=== corr_id AUROC (95% bootstrap CI) ===
Combiner:    0.8841  [0.8664, 0.9005]
Trust Score: 0.9999  [0.9996, 1.0000]
ODIN:        0.6140  [0.5769, 0.6502]

=== DeLong paired tests vs. combiner ===
Combiner vs. Trust Score: diff=-0.1158  z=-13.204  p=8.28e-40
Combiner vs. ODIN:        diff=+0.2701  z=14.502  p=1.18e-47


### Investigating Trust Score's near-perfect 0.9999 `corr_id` AUROC before trusting it

A number this close to 1.0 deserves direct investigation before being reported, not just cited. `TrustScorer`
was fit only on `combiner_fit` (Imagenette's 10 synsets), but `id_test`'s predictions come from the full
1000-way ResNet-50 head — per `trust_score.py`'s own documented edge case, a predicted class outside the 10
fitted classes has no reference density and scores exactly 0 (maximally untrustworthy). Checking directly
whether this mechanical rule, not genuine feature-space reasoning, is what's driving the near-perfect
separation:

In [3]:
pred_test = logits[m_test].argmax(dim=-1)
fitted_classes = set(trust_scorer.classes.tolist())
incorrect_preds = pred_test[~correct_test]
n_incorrect = len(incorrect_preds)
n_outside_fitted = sum(1 for p in incorrect_preds.tolist() if p not in fitted_classes)

print(f"incorrect id_test predictions: {n_incorrect}")
print(f"  predicted class OUTSIDE the 10 fitted Imagenette classes: {n_outside_fitted} ({100*n_outside_fitted/n_incorrect:.1f}%)")
print(f"  predicted class is one of the 10 fitted classes, just the wrong one: {n_incorrect - n_outside_fitted}")
print()
print(f"Trust Score range for CORRECT predictions: [{s_trust_test[correct_test].min():.4f}, {s_trust_test[correct_test].max():.4f}]")
print(f"Trust Score range for INCORRECT predictions: [{s_trust_test[~correct_test].min():.4f}, {s_trust_test[~correct_test].max():.4f}]")

incorrect id_test predictions: 286
  predicted class OUTSIDE the 10 fitted Imagenette classes: 285 (99.7%)
  predicted class is one of the 10 fitted classes, just the wrong one: 1

Trust Score range for CORRECT predictions: [0.9405, 4.0303]
Trust Score range for INCORRECT predictions: [0.0000, 1.0728]


**Real finding, disclosed rather than left implicit in the headline number:** 99.7% of `id_test`'s
incorrect predictions land on a class *outside* the 10 fitted Imagenette classes — and since `id_test`'s
ground-truth labels are themselves restricted to exactly those 10 classes by construction, "predicted class
outside the fitted set" is (for this specific split design) very nearly synonymous with "incorrect,"
triggering the score-0 rule almost every time a prediction is wrong. The near-perfect 0.9999 AUROC is
therefore driven overwhelmingly by this split-design artifact — the model's predicted class either
belonging or not belonging to the plausible label set — not by genuine feature-space typicality reasoning of
the kind `ood_o`/`shift_a` below actually exercise (where every image, correct or not, still gets a real,
non-degenerate score). This is a real, legitimate signal in its own right (a predicted class outside a
known-plausible set is a defensible reliability red flag), but it is a much narrower, more mechanical thing
than "Trust Score is an almost-perfect correctness detector in general," and is reported with that
distinction attached rather than as an inflated headline.

## Comparison 2 — `ood_o`: `id_test` vs. `imagenet_o`

Independent bootstrap CIs per method (not DeLong — while the same *images* underlie every method's
`id_test`/`imagenet_o` groups, this axis has no single shared "correct" binary label the way `corr_id`
does; comparing separation strength via non-overlapping CIs, the same convention `notebooks/18` uses for
its own OOD-suite-vs-baseline comparisons, is the appropriate tool here).

In [4]:
ci_combiner_o = bootstrap_auroc_ci(s_combiner[m_test], s_combiner[m_o], n_bootstrap=3000, seed=0)
ci_trust_o = bootstrap_auroc_ci(s_trust[m_test], s_trust[m_o], n_bootstrap=3000, seed=0)
ci_odin_o = bootstrap_auroc_ci(s_odin[m_test], s_odin[m_o], n_bootstrap=3000, seed=0)

print("=== ood_o AUROC (95% bootstrap CI), id_test vs. imagenet_o ===")
print(f"Combiner:    {ci_combiner_o.auroc:.4f}  [{ci_combiner_o.ci_lo:.4f}, {ci_combiner_o.ci_hi:.4f}]")
print(f"Trust Score: {ci_trust_o.auroc:.4f}  [{ci_trust_o.ci_lo:.4f}, {ci_trust_o.ci_hi:.4f}]")
print(f"ODIN:        {ci_odin_o.auroc:.4f}  [{ci_odin_o.ci_lo:.4f}, {ci_odin_o.ci_hi:.4f}]  <- near/below chance, see the T=1000 finding above")

=== ood_o AUROC (95% bootstrap CI), id_test vs. imagenet_o ===
Combiner:    0.5673  [0.5475, 0.5876]
Trust Score: 0.9013  [0.8911, 0.9117]
ODIN:        0.4645  [0.4454, 0.4842]  <- near/below chance, see the T=1000 finding above


## Comparison 3 — `shift_a`: `id_test` vs. `imagenet_a` (does each signal notice imagenet_a looks unusual at all)

In [5]:
ci_combiner_a = bootstrap_auroc_ci(s_combiner[m_test], s_combiner[m_a], n_bootstrap=3000, seed=0)
ci_trust_a = bootstrap_auroc_ci(s_trust[m_test], s_trust[m_a], n_bootstrap=3000, seed=0)
ci_odin_a = bootstrap_auroc_ci(s_odin[m_test], s_odin[m_a], n_bootstrap=3000, seed=0)

print("=== shift_a AUROC (95% bootstrap CI), id_test vs. imagenet_a ===")
print(f"Combiner:    {ci_combiner_a.auroc:.4f}  [{ci_combiner_a.ci_lo:.4f}, {ci_combiner_a.ci_hi:.4f}]")
print(f"Trust Score: {ci_trust_a.auroc:.4f}  [{ci_trust_a.ci_lo:.4f}, {ci_trust_a.ci_hi:.4f}]")
print(f"ODIN:        {ci_odin_a.auroc:.4f}  [{ci_odin_a.ci_lo:.4f}, {ci_odin_a.ci_hi:.4f}]")

=== shift_a AUROC (95% bootstrap CI), id_test vs. imagenet_a ===
Combiner:    0.8309  [0.8172, 0.8444]
Trust Score: 0.9028  [0.8924, 0.9128]
ODIN:        0.5278  [0.5086, 0.5466]


## Comparison 4 — `corr_a`: within `imagenet_a`, does each signal separate correct from incorrect predictions?

The axis `DESIGN.md` 20.2 and `mahalanobis.py`'s own tests already found near-chance for every
logit-only AND feature-space signal tried so far — checked here for Trust Score and ODIN too, for
completeness, not because a different result was expected.

In [6]:
correct_a = logits[m_a].argmax(dim=-1) == labels[m_a]
s_combiner_a_split = s_combiner[m_a]
s_trust_a_split = s_trust[m_a]
s_odin_a_split = s_odin[m_a]

a_combiner_corr = auroc(s_combiner_a_split[correct_a], s_combiner_a_split[~correct_a])
a_trust_corr = auroc(s_trust_a_split[correct_a], s_trust_a_split[~correct_a])
a_odin_corr = auroc(s_odin_a_split[correct_a], s_odin_a_split[~correct_a])

print("=== corr_a AUROC (point estimate), correct vs. incorrect within imagenet_a ===")
print(f"Combiner:    {a_combiner_corr:.4f}")
print(f"Trust Score: {a_trust_corr:.4f}")
print(f"ODIN:        {a_odin_corr:.4f}")

=== corr_a AUROC (point estimate), correct vs. incorrect within imagenet_a ===
Combiner:    0.5445
Trust Score: 0.4975
ODIN:        0.3677


## Result, stated plainly

Real, same-protocol numbers now exist for ODIN and Trust Score against this project's own combiner,
replacing `STUDY_PLAN.md` 3.6's qualitative citations — reported exactly as found, whichever way each
one landed, and feeding directly into that table's update.

In [7]:
print("Summary — real ResNet-50, same protocol for all three methods")
print()
print(f"{'Axis':<10} {'Combiner':<12} {'Trust Score':<12} {'ODIN':<12}")
print(f"{'corr_id':<10} {ci_combiner.auroc:<12.4f} {ci_trust.auroc:<12.4f} {ci_odin.auroc:<12.4f}")
print(f"{'ood_o':<10} {ci_combiner_o.auroc:<12.4f} {ci_trust_o.auroc:<12.4f} {ci_odin_o.auroc:<12.4f}")
print(f"{'shift_a':<10} {ci_combiner_a.auroc:<12.4f} {ci_trust_a.auroc:<12.4f} {ci_odin_a.auroc:<12.4f}")
print(f"{'corr_a':<10} {a_combiner_corr:<12.4f} {a_trust_corr:<12.4f} {a_odin_corr:<12.4f}")

Summary — real ResNet-50, same protocol for all three methods

Axis       Combiner     Trust Score  ODIN        
corr_id    0.8841       0.9999       0.6140      
ood_o      0.5673       0.9013       0.4645      
shift_a    0.8309       0.9028       0.5278      
corr_a     0.5445       0.4975       0.3677      
